<a href="https://colab.research.google.com/github/AK-SHELL/K-Mean-Customer-Review-Clustering/blob/main/Customer_Review_Clustering_Graph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Ensure NLTK resources are downloaded (only needs to be run once)
try:
    nltk.data.find('corpora/stopwords')
# Catch LookupError which is raised when a resource is not found
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')
try:
    nltk.data.find('corpora/omw-1.4')
except LookupError:
    nltk.download('omw-1.4')


# Here are the csv file.
csv_files = ['Amazon shopping.csv' ]

all_reviews_list = [] # List to store DataFrames from each CSV
df = pd.DataFrame() # Initialize df to an empty DataFrame to prevent NameError

try:
    for file in csv_files:
        try:
            # Attempt to read the CSV
            # You can add encoding='utf-8' or other parameters here if needed
            current_df = pd.read_csv(file)
            print(f"--- Loaded data from '{file}' ---")

            if 'review_text' not in current_df.columns:
                 print(f"Error: 'review_text' column not found in {file}. Please ensure your CSV has a review text column and rename it if necessary.")
                 continue # Skip this file

            # Add a source column to keep track of which file the review came from (optional but helpful)
            current_df['source_file'] = file

            # Add a unique review_id if it doesn't exist or is not unique
            if 'review_id' not in current_df.columns or current_df['review_id'].duplicated().any():
                 # Generate unique IDs based on the index after concatenation
                 print(f"Warning: 'review_id' not found or not unique in {file}. Will generate new IDs after concatenation.")

            all_reviews_list.append(current_df)

        except FileNotFoundError:
            print(f"--- '{file}' not found. Skipping. ---")
        except Exception as e:
            print(f"--- An error occurred while loading '{file}': {e} ---")
            # Catching general exceptions here might make it hard to debug
            # specific CSV reading issues. You might want more specific except blocks.


    # Concatenate all loaded DataFrames
    if all_reviews_list:
        df = pd.concat(all_reviews_list, ignore_index=True)
        print("--- Successfully loaded and combined data from specified CSV files ---")
         # Generate unique review_ids if they were not present or not unique
        if 'review_id' not in df.columns or df['review_id'].duplicated().any():
             df['review_id'] = range(1, len(df) + 1)
             print("Generated unique 'review_id' for all combined reviews.")

except Exception as e:
    print(f"An unexpected error occurred during data loading: {e}")


if not df.empty:
    print("--- Data Head (first 5 rows) ---")
    print(df.head())
    print("\n")

    # Text Preprocessing
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()

    def preprocess_text(text):
        # Ensure the input is a string to prevent errors if there are non-string entries
        if not isinstance(text, str):
            return ""
        text = text.lower() # Convert to lowercase
        text = re.sub(r'[^a-z\s]', '', text) # Remove non-alphabetic characters
        words = text.split() # Split into words
        words = [word for word in words if word not in stop_words] # Remove stop words
        words = [lemmatizer.lemmatize(word) for word in words] # Lemmatize words
        return ' '.join(words)

    # Apply preprocessing to the 'review_text' column
    if 'review_text' in df.columns:
        df['processed_review'] = df['review_text'].apply(preprocess_text)
        print("--- Processed Reviews Head ---")
        print(df[['review_text', 'processed_review']].head())
        print("\n")
    else:
        print("Error: 'review_text' column not found after data loading. Cannot proceed with preprocessing and analysis.")


    # Continue with the rest of the script only if 'processed_review' column exists
    if 'processed_review' in df.columns and not df.empty:

        # Feature Extraction (TF-IDF Vectorization)

        vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
        X = vectorizer.fit_transform(df['processed_review'])
        print(f"--- TF-IDF Matrix Shape: {X.shape} ---")
        print("This means we have reviews (samples) and features (words).\n")

        # Clustering using K-Means
        num_clusters = 4
        if len(df) < num_clusters:
            print(f"Warning: Number of samples ({len(df)}) is less than the desired number of clusters ({num_clusters}). Adjusting number of clusters.")
            num_clusters = max(1, len(df) - 1) # Set to max(1, num_samples - 1)
            print(f"New number of clusters: {num_clusters}")

        if num_clusters > 0:
            kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
            df['cluster'] = kmeans.fit_predict(X)
            print("--- Reviews with Assigned Clusters ---")
            print(df[['review_text', 'cluster']].head())
            print("\n")

            #  Dimensionality Reduction using t-SNE
            actual_samples = len(df)
            current_perplexity = min(30, actual_samples - 1)
            if actual_samples < 5:
                print(f"Warning: Dataset is too small for meaningful t-SNE visualization ({actual_samples} samples).")
                print("Adjusting perplexity to max(1, actual_samples - 1). Consider providing more data for better results.")
                current_perplexity = max(1, actual_samples - 1)
            elif actual_samples < 30:
                 print(f"Warning: Dataset has less than 30 samples ({actual_samples}). Adjusting perplexity.")
                 current_perplexity = actual_samples - 1


            if actual_samples > 1:
                tsne = TSNE(n_components=2, random_state=42, perplexity=current_perplexity, n_iter=1000)
                X_tsne = tsne.fit_transform(X.toarray())

                # Add t-SNE components to the DataFrame for plotting
                df['tsne_x'] = X_tsne[:, 0]
                df['tsne_y'] = X_tsne[:, 1]
                print("--- t-SNE Coordinates Head ---")
                # Ensure review_id exists before trying to print it
                cols_to_print = ['tsne_x', 'tsne_y', 'cluster']
                if 'review_id' in df.columns:
                     cols_to_print.insert(0, 'review_id')
                print(df[cols_to_print].head())
                print("\n")

                # Visualization using Seaborn and Matplotlib
                plt.figure(figsize=(10, 8))
                sns.scatterplot(
                    x='tsne_x',
                    y='tsne_y',
                    hue='cluster',
                    palette='viridis',
                    data=df,
                    legend='full',
                    alpha=0.7
                )
                plt.title(f'Customer Review Clusters (K-Means with {num_clusters} Clusters via t-SNE)', fontsize=16)
                plt.xlabel('t-SNE Component 1', fontsize=12)
                plt.ylabel('t-SNE Component 2', fontsize=12)
                plt.grid(True, linestyle='--', alpha=0.6)

                # Annotate points with review_id for easier inspection (optional)
                # This can make the plot cluttered for large datasets, consider disabling for many reviews.
                # Only attempt to annotate if review_id exists and the dataset isn't too large
                if 'review_id' in df.columns and len(df) <= 100: # Annotate up to 100 points
                     for i, row in df.iterrows():
                        plt.annotate(row['review_id'], (row['tsne_x'], row['tsne_y']), textcoords="offset points", xytext=(5,5), ha='center', fontsize=8)
                elif len(df) > 100 and 'review_id' in df.columns:
                    print("Skipping annotations as dataset is large.")


                plt.show()

                # --- Display Most Representative Reviews (Centroids) for each Cluster ---
                print("--- Top Reviews per Cluster (Closest to Centroid) ---")
                for i in range(num_clusters):
                    cluster_reviews = df[df['cluster'] == i]
                    if not cluster_reviews.empty:
                        # Calculate distances from each review in the cluster to its centroid
                        cluster_indices = cluster_reviews.index
                        distances = np.linalg.norm(X[cluster_indices] - kmeans.cluster_centers_[i], axis=1)
                        # Get the index *within the cluster_reviews DataFrame* of the closest review
                        closest_review_in_cluster_idx = np.argmin(distances)
                        # Get the original index *in the main df DataFrame*
                        closest_review_original_idx = cluster_reviews.iloc[closest_review_in_cluster_idx].name

                        # Ensure review_id and review_text columns exist before accessing
                        review_id_val = df.loc[closest_review_original_idx, 'review_id'] if 'review_id' in df.columns else 'N/A'
                        review_text_val = df.loc[closest_review_original_idx, 'review_text'] if 'review_text' in df.columns else 'Review text not available.'

                        print(f"\nCluster {i} (Count: {len(cluster_reviews)}):")
                        print(f"  Representative Review (ID {review_id_val}): \"{review_text_val}\"")
                    else:
                        print(f"\nCluster {i}: No reviews in this cluster.")
            else:
                 print("Skipping t-SNE visualization and centroid review display as there are not enough samples (> 1).")
        else:
            print("Skipping clustering, t-SNE, and centroid review display as the number of clusters is zero or less.")

    else:
        print("Dataframe is empty or 'processed_review' column is missing. Analysis stopped.")

else:
    print("No data was successfully loaded from the specified CSV files. Analysis stopped.")

# Text Preprocessing
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Ensure the input is a string to prevent errors if there are non-string entries
    if not isinstance(text, str):
        return ""
    text = text.lower() # Convert to lowercase
    text = re.sub(r'[^a-z\s]', '', text) # Remove non-alphabetic characters
    words = text.split() # Split into words
    words = [word for word in words if word not in stop_words] # Remove stop words
    words = [lemmatizer.lemmatize(word) for word in words] # Lemmatize words
    return ' '.join(words)

# Apply preprocessing to the 'review_text' column
if 'review_text' in df.columns:
    df['processed_review'] = df['review_text'].apply(preprocess_text)
    print("--- Processed Reviews Head ---")
    print(df[['review_text', 'processed_review']].head())
    print("\n")
else:
    print("Error: 'review_text' column not found after data loading. Cannot proceed with preprocessing and analysis.")


# Continue with the rest of the script only if 'processed_review' column exists
if 'processed_review' in df.columns and not df.empty:

    # Feature Extraction (TF-IDF Vectorization)

    vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
    X = vectorizer.fit_transform(df['processed_review'])
    print(f"--- TF-IDF Matrix Shape: {X.shape} ---")
    print("This means we have reviews (samples) and features (words).\n")

    # Clustering using K-Means
    num_clusters = 4
    if len(df) < num_clusters:
        print(f"Warning: Number of samples ({len(df)}) is less than the desired number of clusters ({num_clusters}). Adjusting number of clusters.")
        num_clusters = max(1, len(df) - 1) # Set to max(1, num_samples - 1)
        print(f"New number of clusters: {num_clusters}")

    if num_clusters > 0:
        kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
        df['cluster'] = kmeans.fit_predict(X)
        print("--- Reviews with Assigned Clusters ---")
        print(df[['review_text', 'cluster']].head())
        print("\n")

        #  Dimensionality Reduction using t-SNE
        actual_samples = len(df)
        current_perplexity = min(30, actual_samples - 1)
        if actual_samples < 5:
            print(f"Warning: Dataset is too small for meaningful t-SNE visualization ({actual_samples} samples).")
            print("Adjusting perplexity to max(1, actual_samples - 1). Consider providing more data for better results.")
            current_perplexity = max(1, actual_samples - 1)
        elif actual_samples < 30:
             print(f"Warning: Dataset has less than 30 samples ({actual_samples}). Adjusting perplexity.")
             current_perplexity = actual_samples - 1


        if actual_samples > 1:
            tsne = TSNE(n_components=2, random_state=42, perplexity=current_perplexity, n_iter=1000)
            X_tsne = tsne.fit_transform(X.toarray())

            # Add t-SNE components to the DataFrame for plotting
            df['tsne_x'] = X_tsne[:, 0]
            df['tsne_y'] = X_tsne[:, 1]
            print("--- t-SNE Coordinates Head ---")
            # Ensure review_id exists before trying to print it
            cols_to_print = ['tsne_x', 'tsne_y', 'cluster']
            if 'review_id' in df.columns:
                 cols_to_print.insert(0, 'review_id')
            print(df[cols_to_print].head())
            print("\n")

            # Visualization using Seaborn and Matplotlib
            plt.figure(figsize=(10, 8))
            sns.scatterplot(
                x='tsne_x',
                y='tsne_y',
                hue='cluster',
                palette='viridis',
                data=df,
                legend='full',
                alpha=0.7
            )
            plt.title(f'Customer Review Clusters (K-Means with {num_clusters} Clusters via t-SNE)', fontsize=16)
            plt.xlabel('t-SNE Component 1', fontsize=12)
            plt.ylabel('t-SNE Component 2', fontsize=12)
            plt.grid(True, linestyle='--', alpha=0.6)

            # Annotate points with review_id for easier inspection (optional)
            # This can make the plot cluttered for large datasets, consider disabling for many reviews.
            # Only attempt to annotate if review_id exists and the dataset isn't too large
            if 'review_id' in df.columns and len(df) <= 100: # Annotate up to 100 points
                 for i, row in df.iterrows():
                    plt.annotate(row['review_id'], (row['tsne_x'], row['tsne_y']), textcoords="offset points", xytext=(5,5), ha='center', fontsize=8)
            elif len(df) > 100 and 'review_id' in df.columns:
                print("Skipping annotations as dataset is large.")


            plt.show()

            # --- Display Most Representative Reviews (Centroids) for each Cluster ---
            print("--- Top Reviews per Cluster (Closest to Centroid) ---")
            for i in range(num_clusters):
                cluster_reviews = df[df['cluster'] == i]
                if not cluster_reviews.empty:
                    # Calculate distances from each review in the cluster to its centroid
                    cluster_indices = cluster_reviews.index
                    distances = np.linalg.norm(X[cluster_indices] - kmeans.cluster_centers_[i], axis=1)
                    # Get the index *within the cluster_reviews DataFrame* of the closest review
                    closest_review_in_cluster_idx = np.argmin(distances)
                    # Get the original index *in the main df DataFrame*
                    closest_review_original_idx = cluster_reviews.iloc[closest_review_in_cluster_idx].name

                    # Ensure review_id and review_text columns exist before accessing
                    review_id_val = df.loc[closest_review_original_idx, 'review_id'] if 'review_id' in df.columns else 'N/A'
                    review_text_val = df.loc[closest_review_original_idx, 'review_text'] if 'review_text' in df.columns else 'Review text not available.'

                    print(f"\nCluster {i} (Count: {len(cluster_reviews)}):")
                    print(f"  Representative Review (ID {review_id_val}): \"{review_text_val}\"")
                else:
                    print(f"\nCluster {i}: No reviews in this cluster.")
        else:
             print("Skipping t-SNE visualization and centroid review display as there are not enough samples (> 1).")
    else:
        print("Skipping clustering, t-SNE, and centroid review display as the number of clusters is zero or less.")

else:
    print("Dataframe is empty or 'processed_review' column is missing. Analysis stopped.")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


--- 'Amazon shopping.csv' not found. Skipping. ---


NameError: name 'df' is not defined